# Spec-Driven Pipeline with Code Extensions

[![Open in Colab](https://img.shields.io/badge/Open_in_Colab-F9AB00?style=for-the-badge&logo=googlecolab&logoColor=white)](https://colab.research.google.com/github/RealSlimMahdi/fleet-cdisc/blob/main/docs/examples/02_spec_driven_pipeline.ipynb)

> **Interactive Playground** — Click the badge above to launch this notebook in Google Colab and experiment with Fleet live, no local setup needed.

This notebook demonstrates the standard workflow for Fleet in a production environment: **Spec-First**.

1. You define as much as possible in `define.yaml`.
2. Fleet automatically executes it.
3. You write a thin layer of Python code for complex custom rules.

In [1]:
import importlib, subprocess, sys

if importlib.util.find_spec("fleet") is None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q",
         "fleet-cdisc@git+https://github.com/RealSlimMahdi/fleet-cdisc.git"]
    )

In [2]:
import polars as pl
import fleet
from fleet import load_study_context
# Normally, we just build context:
# ctx = load_study_context('specs/cdiscpilot01/adam/define.yaml')
# adsl = ctx.build('ADSL')

print("Loaded Spec context successfully.")

Loaded Spec context successfully.


## Extending the Built Data

Suppose `define.yaml` generated our base ADSL pipeline, but we have a completely proprietary calculation that the specification does not natively handle via its algorithms. We simply extend the resulting dataframe!

In [3]:
# Mocking the dataframe that would have been returned by ctx.build('ADSL')
base_adsl = pl.DataFrame({
    "STUDYID": ["MYSTUDY"] * 2,
    "USUBJID": ["S001", "S002"],
    "TRTSDT": ["2023-01-01", "2023-01-05"],
    "TRTEDT": ["2023-01-10", "2023-01-20"]
})

# Adding our "Thin layer of code":
final_adsl = (
    base_adsl.adam.adsl
    .source("USUBJID")
    .duration("TRTDUR", start_date="TRTSDT", end_date="TRTEDT")
    .derivation("COMPLEX_FLAG", expression=pl.when(pl.col("TRTDUR") > 10).then(pl.lit("Y")).otherwise(pl.lit(None)))
    .to_frame()
)

print(final_adsl)

shape: (2, 6)
┌─────────┬─────────┬────────────┬────────────┬────────┬──────────────┐
│ STUDYID ┆ USUBJID ┆ TRTSDT     ┆ TRTEDT     ┆ TRTDUR ┆ COMPLEX_FLAG │
│ ---     ┆ ---     ┆ ---        ┆ ---        ┆ ---    ┆ ---          │
│ str     ┆ str     ┆ str        ┆ str        ┆ i64    ┆ str          │
╞═════════╪═════════╪════════════╪════════════╪════════╪══════════════╡
│ MYSTUDY ┆ S001    ┆ 2023-01-01 ┆ 2023-01-10 ┆ 10     ┆ null         │
│ MYSTUDY ┆ S002    ┆ 2023-01-05 ┆ 2023-01-20 ┆ 16     ┆ Y            │
└─────────┴─────────┴────────────┴────────────┴────────┴──────────────┘
